# Company Baseline Evaluation

1. The ner_pipeline script cannot be shared as it is the IBFD's code.
2. Claude was used to adapt their files since it was a bit complex

In [ ]:
import json
from pathlib import Path
import spacy
from seqeval.metrics import classification_report
from seqeval.scheme import IOB2

from ner_pipeline import (
    ALLOWED_SPACY_LABELS,
    COURT_PATTERN_COMPILED,
    PROVISION_PATTERN_COMPILED,
    UNIFIED_LABEL2ID,
)

test_set       = r"/Users/manyawalavalkar/Desktop/thesis_files/thesis-ner-manya-explore/datasets/my_data/final_dataset_1st_may.conll"
comp_gaz_path   = (r"/Users/manyawalavalkar/Desktop/thesis_files/ner_annotation/gazetteer_enriched.json")
spacy_model      = 'en_core_web_sm'
tax_concept_gaz = "TAX_CONCEPT.txt"
tax_type_gaz = "TAX_TYPE.txt"
jurisdiction_gaz = r"/Users/manyawalavalkar/Desktop/thesis_files/ner_annotation/true_ONLY_JUR.json"


## Loading the test set

In [ ]:
sentences = parse_conll(Path(test_set))

gold_tokens = []
gold_labels = []

for sentence in sentences:
    tokens = []
    labels = []

    for token, label in sentence:
        tokens.append(token)
        labels.append(label)

    gold_tokens.append(tokens)
    gold_labels.append(labels)

## Loading the gazetteers

In [ ]:
def load_gazetteer(path):
    """loads a JSON gazetteer to compile."""
    with open(path, encoding="utf-8") as file:
        return json.load(file)
company_gaz = load_gazetteer(comp_gaz_path)
heuristics_gaz = []
# adding tax concepts
with open(tax_concept_gaz, encoding="utf-8") as file:
    for line in file:
        term = line.strip()
        if term:
            heuristics_gaz.append({"surface_forms": [term], "entity_type": "TAX_CONCEPT"})
# adding tax types
with open(tax_type_gaz, encoding="utf-8") as file:
    for line in file:
        term = line.strip()
        if term:
            heuristics_gaz.append({"surface_forms": [term], "entity_type": "TAX_TYPE"})
# adding jurisdictions
with open(jurisdiction_gaz, encoding="utf-8") as file:
    jurisdiction_entries = json.load(file)
heuristics_gaz.extend(jurisdiction_entries)

## Gazetteer label mapping

In [19]:
GAZETTEER_TYPE_MAPPING = {
    "TAX_TYPE": "TAX_TYPE",
    "TAX_CONCEPT": "TAX_CONCEPT",
    "TAX_RELIEF": "TAX_CONCEPT",
    "TAX_BASE": "TAX_CONCEPT",
    "LEGAL_ENTITY": "TAX_CONCEPT",
    "TAX_PROCEDURE": "TAX_CONCEPT",
    "TAX_AGREEMENT": "TAX_CONCEPT",
    "TAX_AUTHORITY": "TAX_CONCEPT",
    "ANTI_AVOIDANCE": "TAX_CONCEPT",
    "TRANSFER_PRICING": "TAX_CONCEPT",
    "JURISDICTION": "JURISDICTION",
}

## Find gazetteer matches

In [ ]:
def find_gazetteer_matches(tokens, gazetteer):
    """
finds gaz terms in tokenized sentence - adapted from company file
    """
    matches = []
    # create a single string so that multi-word gazetteer entries can be matched.
    text = " ".join(tokens)
    text_lower = text.lower()
    # store surface forms together with their entries.
    surface_forms = {}
    for entry in gazetteer:
        for surface_form in entry.get("surface_forms", []):
            surface = surface_form.lower().strip()
            if surface and len(surface) > 1:
                if surface not in surface_forms:
                    surface_forms[surface] = entry
                    
    # check longer terms first since longer matches take priority
    surfaces = sorted(surface_forms.keys(), key=len, reverse=True)
    matched_positions = set()
    for surface in surfaces:
        # ignore very short terms
        if len(surface) < 3:
            continue
        start = 0
        while True:
            #find the next surface form occurrence
            index = text_lower.find(surface, start)
            if index == -1:
                break
            end_index = index + len(surface)
            # make sure the match is a complete word or phrase rather than part of a larger word
            left_is_letter = (index > 0 and text_lower[index - 1].isalnum())
            right_is_letter = (end_index < len(text_lower) and text_lower[end_index].isalnum())
            if left_is_letter or right_is_letter:
                start = index + 1
                continue
            #converting the character spans back to tok positions
            token_start, token_end = character_2_token(tokens, index, end_index)
            if token_start is not None:
                match_positions = set(range(token_start, token_end))
                # not adding any overlapping matches
                if not match_positions.intersection(matched_positions):
                    entry = surface_forms[surface]
                    original_type = entry.get("entity_type","TAX_CONCEPT" )
                    #map the gaz original label to the label used for eval
                    label = GAZETTEER_TYPE_MAPPING.get(original_type, "TAX_CONCEPT") #fall back for the company gaz since some terms arent labeled
                    matches.append({"text": " ".join(tokens[token_start:token_end]),
                        "label": label, "original_label": original_type, "term_id": entry.get("term_id"), "token_start": token_start, "token_end": token_end, "source": "gazetteer"})
                    matched_positions.update(match_positions)
            start = index + 1
    #return matches in their original sentence order
    matches.sort(key=lambda match: match["token_start"])
    return matches

Convert character positions to token positions

In [ ]:
def character_2_token(tokens, char_start, char_end):
    """this converts char spans into token postions"""
    position = 0
    token_start = None
    token_end = None
    for i, token in enumerate(tokens):
        token_start_char = position 
        token_end_char = position + len(token)
        if (token_start_char <= char_start < token_end_char): #check if the start of the char span falls within the current token range
            token_start = i
        if (token_start_char < char_end <= token_end_char): #check if the end also falls within the token span
            token_end = i + 1 
        position = token_end_char + 1
    return token_start, token_end #return the token position range of the char span

## Run SpaCy and RegEx

In [ ]:
def find_spacy_entities(tokens, use_regex):
    """this runs spacy and returns the entities we want"""
    text = " ".join(tokens)
    doc = nlp(text) #spacy object
    entities = []
    for entity in doc.ents: #for all entities
        if entity.label_ not in ALLOWED_SPACY_LABELS: #keep only allowed ones
            continue
        label = entity.label_ #get label
        # Using regex rules from the ibfd files
        if use_regex: #then use regex if activated
            if (label == "ORG" and COURT_PATTERN_COMPILED.search(entity.text)):
                label = "COURT"
            if (label == "LAW" and PROVISION_PATTERN_COMPILED.search(entity.text)):
                label = "PROVISION"
        token_start, token_end = character_2_token(tokens, entity.start_char, entity.end_char)
        if token_start is not None:
            entities.append({"label": label, "token_start": token_start, "token_end": token_end})

    return entities


## Combine SpaCy and Gaz matches

In [ ]:
def merge_entities(spacy_entities, gazetteer_matches):
    """this combines spacy and gaz entities but spacy preds have precedence"""
    used_tokens = set() #to keep track of already label assigned tokens
    for entity in spacy_entities:
        for i in range(entity["token_start"],entity["token_end"]):
            used_tokens.add(i)
    entities = list(spacy_entities) #get all spacy ents
    for match in gazetteer_matches: #get the token positions covered by gaz matches
        match_tokens = set(range(match["token_start"], match["token_end"]))
        if not match_tokens.intersection(used_tokens): #only add gaz match if it does not overlap with spacy entity
            entities.append(match)
            used_tokens.update(match_tokens)
    entities.sort(key=lambda entity: entity["token_start"]) #return in original sentence order
    return entities

## Convert entities to BIO labels

In [ ]:
def entities_to_bio(tokens, entities):
    """this converts entity spans into bio labels."""
    labels = ["O"] * len(tokens)
    for entity in entities:
        label = entity["label"] #get the label
        if "B-" + label not in UNIFIED_LABEL2ID: #see if its in the allowed labels
            continue
        start = entity["token_start"]  #get start and end position of the entity
        end = entity["token_end"]
        labels[start] = "B-" + label # B- label to the first oken
        for i in range(start + 1, end): #assign I- labels to all remaining tokens in the entity
            labels[i] = "I-" + label

    return labels


## predict one sentence

In [ ]:
def predict_sentence(tokens, gazetteer=None, use_spacy=True, use_regex=False, use_gazetteer=False):
    """this predicts bio labels for one sent"""
    if not tokens: 
        return []
    spacy_entities = []
    if use_spacy: 
        spacy_entities = find_spacy_entities(tokens, use_regex) #use spacy+regex
    gazetteer_matches = []
    if use_gazetteer:
        gazetteer_matches = find_gazetteer_matches(tokens, gazetteer) #run gaz matching
    entities = merge_entities(spacy_entities, gazetteer_matches) #combine predictions, give spacy ents precedence
    return entities_to_bio(tokens, entities) #then we can convert to bio labels

def predict_all(tokens_list, gazetteer=None, use_spacy=True, use_regex=False, use_gazetteer=False): 
    """predicts bio labs for all sentences"""
    predictions = []
    for tokens in tokens_list:
        prediction = predict_sentence(tokens, gazetteer=gazetteer, use_spacy=use_spacy, use_regex=use_regex, use_gazetteer=use_gazetteer)
        predictions.append(prediction)
    return predictions


## Save predictions and eval

In [ ]:
def write_conll(path, tokens_list, labels_list):
    """this saves preds to conll"""
    with open(path, "w", encoding="utf-8") as file:
        for tokens, labels in zip(tokens_list, labels_list):
            for token, label in zip(tokens, labels):
                file.write(token + "\t" + label + "\n")
            file.write("\n")

def evaluate(gold_labels, predictions):
    """eval with strict seqeval"""
    print(classification_report(gold_labels, predictions, mode="strict", scheme=IOB2, digits=4))


# RUN!!

In [ ]:
experiments = {
    "spacy": {
        "gazetteer": None,
        "use_spacy": True,
        "use_regex": False,
        "use_gazetteer": False
    },

    "spacy + regex": {
        "gazetteer": None,
        "use_spacy": True,
        "use_regex": True,
        "use_gazetteer": False
    },

    "spacy + regex + company gaz": {
        "gazetteer": company_gaz,
        "use_spacy": True,
        "use_regex": True,
        "use_gazetteer": True
    },

    "spacy + regex + my gaz": {
        "gazetteer": heuristics_gaz,
        "use_spacy": True,
        "use_regex": True,
        "use_gazetteer": True
    },

    "company gaz only": {
        "gazetteer": company_gaz,
        "use_spacy": False,
        "use_regex": False,
        "use_gazetteer": True
    },

    "my gaz only": {
        "gazetteer": heuristics_gaz,
        "use_spacy": False,
        "use_regex": False,
        "use_gazetteer": True
    }
}

for name, experiment in experiments.items():

    print(name)
    print("\n")
    predictions = predict_all(gold_tokens, gazetteer=experiment["gazetteer"], use_spacy=experiment["use_spacy"],use_regex=experiment["use_regex"], use_gazetteer=experiment["use_gazetteer"])
    filename = ("pred_"+ name.replace(" ", "_").replace("+", "plus")+".conll")
    write_conll(filename, gold_tokens, predictions)
    evaluate(gold_labels, predictions)


spacy
              precision    recall  f1-score   support

       COURT     0.0000    0.0000    0.0000       106
        DATE     0.5091    0.6364    0.5657       132
         GPE     0.6807    0.6488    0.6644       299
JURISDICTION     0.0000    0.0000    0.0000       153
         LAW     0.0094    0.0154    0.0117        65
         ORG     0.1350    0.3575    0.1960       221
      PERSON     0.5301    0.5215    0.5257       186
   PROVISION     0.0000    0.0000    0.0000        87
 TAX_CONCEPT     0.0000    0.0000    0.0000       353
    TAX_TYPE     0.0000    0.0000    0.0000        86

   micro avg     0.3437    0.2695    0.3021      1688
   macro avg     0.1864    0.2180    0.1964      1688
weighted avg     0.2368    0.2695    0.2460      1688


spacy + regex
              precision    recall  f1-score   support

       COURT     0.3451    0.3679    0.3562       106
        DATE     0.5091    0.6364    0.5657       132
         GPE     0.6807    0.6488    0.6644       299
JU